# 1. Iterative Policy Evaluation
### Evaluates a Given Policy

**Policy Evaluation** computes the state-value function $V^\pi(s)$ for an arbitrary given policy $\pi$. In a known MDP model, this is done iteratively using the **Bellman Expectation Backup Operator**.

#### Mathematical Foundation:
1. **Bellman Expectation Equation for $V^\pi$**:
   $$V^\pi(s) = \sum_{a} \pi(a|s) \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V^\pi(s') \right]$$
2. **Iterative Update Rule**:
   $$V_{k+1}(s) \leftarrow \sum_{a} \pi(a|s) \sum_{s', r} P(s', r | s, a) \left[ r + \gamma V_k(s') \right]$$
3. **Contraction Mapping Theorem**:
   Because the Bellman operator is a $\gamma$-contraction in the maximum norm, the sequence $\{V_k\}$ is guaranteed to converge to the unique true value function $V^\pi$ as $k \to \infty$.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Step 2: Define a 4x4 GridWorld MDP Environment
# States: 16 grid cells (0 to 15), where (0,0) [state 0] and (3,3) [state 15] are terminal
# Actions: 0: UP, 1: RIGHT, 2: DOWN, 3: LEFT

class GridWorld:
    def __init__(self, size=4):
        self.size = size
        self.num_states = size * size
        self.num_actions = 4
        self.actions = ['UP', 'RIGHT', 'DOWN', 'LEFT']
        self.terminal_states = {0, self.num_states - 1}
        
    def get_next_state_and_reward(self, state, action):
        if state in self.terminal_states:
            return state, 0.0
            
        r, c = divmod(state, self.size)
        if action == 0:    # UP
            r = max(r - 1, 0)
        elif action == 1:  # RIGHT
            c = min(c + 1, self.size - 1)
        elif action == 2:  # DOWN
            r = min(r + 1, self.size - 1)
        elif action == 3:  # LEFT
            c = max(c - 1, 0)
            
        next_state = r * self.size + c
        reward = -1.0  # -1 step cost until terminal goal reached
        return next_state, reward

env = GridWorld()
print(f"GridWorld initialized with {env.num_states} states and {env.num_actions} actions.")


In [ ]:
# Step 3: Policy Evaluation Implementation
def policy_evaluation(env, policy, gamma=1.0, theta=1e-4):
    V = np.zeros(env.num_states)
    iterations = 0
    
    while True:
        delta = 0.0
        V_new = np.copy(V)
        for s in range(env.num_states):
            if s in env.terminal_states:
                continue
                
            v_s = 0.0
            for a in range(env.num_actions):
                prob_a = policy[s][a]
                if prob_a > 0:
                    next_s, r = env.get_next_state_and_reward(s, a)
                    v_s += prob_a * (r + gamma * V[next_s])
                    
            V_new[s] = v_s
            delta = max(delta, abs(V[s] - V_new[s]))
            
        V = V_new
        iterations += 1
        if delta < theta:
            break
            
    print(f"Policy Evaluation converged in {iterations} iterations (theta={theta}).")
    return V


In [ ]:
# Step 4: Evaluate Uniform Random Policy
# Under random policy, probability for each action = 1/4 = 0.25
random_policy = np.ones((env.num_states, env.num_actions)) / env.num_actions

V_random = policy_evaluation(env, random_policy, gamma=1.0, theta=1e-4)
print("Computed State-Values for Random Policy:
", V_random.reshape((4, 4)).round(2))


In [ ]:
# Function to plot State-Value Heatmap and Policy Arrows
def plot_grid(V, policy=None, title="State-Value Grid"):
    fig, ax = plt.subplots(figsize=(6, 6))
    grid_v = V.reshape((4, 4))
    
    cax = ax.matshow(grid_v, cmap='Blues_r')
    fig.colorbar(cax)
    
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            val_text = f"{V[s]:.1f}"
            if s in [0, 15]:
                val_text += "\n(Goal)"
            elif policy is not None:
                best_a = np.argmax(policy[s])
                val_text += f"\n{arrows[best_a]}"
                
            ax.text(c, r, val_text, ha='center', va='center', color='black', fontsize=11, weight='bold')
            
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    plt.title(title, pad=20)
    plt.show()


In [ ]:
# Step 6: Visualize State Values for Random Policy
plot_grid(V_random, title="State-Value Function V(s) for Uniform Random Policy")


### Summary
- Evaluates how good a specific policy $\pi$ is.
- Forms the "evaluation" phase inside Generalized Policy Iteration (GPI).
- Computes true discounted expected returns from each starting state under fixed behavior.
